# TatVTON — Colab Component Test Notebook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/khope/tatvton/blob/main/notebooks/tatvton_colab_test.ipynb)

**Purpose**: Test each TatVTON SDK component individually on Colab T4 GPU (16 GB VRAM).  
**Note**: The `khope/tatvton-controlnet-v1` model is not yet on HF Hub.  
Section 4 uses SDXL + IP-Adapter directly (without ControlNet).

---

| Section | Content | GPU |
|---------|---------|-----|
| 0. Setup | GPU check, package install | - |
| 1. Images | Load body / tattoo test images | - |
| 2. SAM2 Mask | `SkinMaskExtractor` with PointPrompt / BBoxPrompt | ~3 GB |
| 3. Mask Processing | `dilate_mask`, `feather_mask` visualization | - |
| 4. SDXL Inpainting | IP-Adapter without ControlNet | ~8 GB |
| 5. Compositing | `Compositor` alpha blending | - |
| 6. Integration | End-to-end pipeline | ~8 GB |

## 0. Environment Setup

In [ ]:
#@title 0a. GPU Check {display-mode: "form"}
import subprocess, torch

result = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
print(result.stdout)

assert torch.cuda.is_available(), (
    "GPU not available! Go to: Runtime > Change runtime type > T4 GPU"
)
gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_mem / 1024**3
print(f"GPU : {gpu_name}")
print(f"VRAM: {vram_gb:.1f} GB")

In [ ]:
#@title 0b. Install packages {display-mode: "form"}
# ── Install TatVTON SDK ──────────────────────────────────────────────
# Option 1: from GitHub (default)
!pip install -q "tatvton @ git+https://github.com/khope/tatvton.git"

# Option 2: from local upload (uncomment if repo is private)
# from google.colab import files
# uploaded = files.upload()          # upload the .whl or .tar.gz
# !pip install -q *.whl

# ── Install SAM 2 (not in tatvton deps) ──────────────────────────────
!pip install -q "sam-2 @ git+https://github.com/facebookresearch/sam2.git"

In [ ]:
#@title 0c. Verify imports {display-mode: "form"}
import tatvton
print(f"tatvton {tatvton.__version__}")

from tatvton import (
    TatVTONConfig, PointPrompt, BBoxPrompt, MaskResult, PipelineOutput,
)
from tatvton.preprocessing.skin_mask_extractor import SkinMaskExtractor
from tatvton.utils.mask import (
    dilate_mask, feather_mask, invert_mask, mask_to_pil, pil_to_mask,
)
from tatvton.utils.image import (
    resize_for_pipeline, pil_to_numpy, numpy_to_pil,
)
from tatvton.postprocessing.compositing import Compositor

print("All imports OK")

In [ ]:
#@title 0d. Helper utilities (run once) {display-mode: "form"}
import gc
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image


def cleanup_vram():
    """Free GPU memory between heavy model sections."""
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        free, total = torch.cuda.mem_get_info()
        print(f"VRAM: {free / 1024**3:.1f} GB free / {total / 1024**3:.1f} GB total")


def show_images(images, titles=None, figsize=(16, 4)):
    """Display images side by side."""
    n = len(images)
    fig, axes = plt.subplots(1, n, figsize=figsize)
    if n == 1:
        axes = [axes]
    for i, (ax, img) in enumerate(zip(axes, images)):
        if isinstance(img, Image.Image):
            ax.imshow(img)
        elif isinstance(img, np.ndarray):
            if img.ndim == 2:
                ax.imshow(img, cmap="gray")
            else:
                ax.imshow(img)
        ax.axis("off")
        if titles and i < len(titles):
            ax.set_title(titles[i])
    plt.tight_layout()
    plt.show()


def show_mask_overlay(image, mask, alpha=0.5, color=(255, 0, 0)):
    """Overlay a boolean mask on an image with a colored tint."""
    img_arr = np.array(image.convert("RGB")).copy()
    overlay = img_arr.copy()
    overlay[mask] = color
    blended = (img_arr * (1 - alpha) + overlay * alpha).astype(np.uint8)
    return Image.fromarray(blended)


print("Helpers loaded.")

## 1. Sample Images

Choose an image source:

- **synthetic** — auto-generated test images (no download, always works)
- **upload** — upload your own body / tattoo photos from local disk
- **url** — download from a public URL (replace the defaults)

In [ ]:
#@title 1. Load Sample Images {display-mode: "form"}
image_source = "synthetic"  #@param ["synthetic", "upload", "url"]

if image_source == "synthetic":
    # -- Synthetic body image (skin-toned with a darker "clothing" band) --
    rng = np.random.RandomState(42)
    body_arr = np.full((768, 512, 3), (210, 180, 160), dtype=np.uint8)
    noise = rng.randint(-15, 15, body_arr.shape, dtype=np.int16)
    body_arr = np.clip(body_arr.astype(np.int16) + noise, 0, 255).astype(np.uint8)
    body_arr[:200, :] = np.clip(
        body_arr[:200, :].astype(np.int16) - 80, 0, 255
    ).astype(np.uint8)
    body_img = Image.fromarray(body_arr)

    # -- Synthetic tattoo design (geometric pattern on white) --
    t_arr = np.full((224, 224, 3), 255, dtype=np.uint8)
    yy, xx = np.mgrid[:224, :224]
    cx, cy = 112, 112
    dist = np.sqrt((xx - cx) ** 2 + (yy - cy) ** 2)
    pattern = (
        ((dist > 30) & (dist < 40))
        | ((dist > 55) & (dist < 65))
        | ((dist > 80) & (dist < 90))
        | ((np.abs(xx - cx) < 3) & (dist < 90))
        | ((np.abs(yy - cy) < 3) & (dist < 90))
    )
    t_arr[pattern] = [0, 0, 0]
    tattoo_img = Image.fromarray(t_arr)

elif image_source == "upload":
    from google.colab import files

    print("Upload body image:")
    up = files.upload()
    body_img = Image.open(list(up.keys())[0]).convert("RGB")

    print("Upload tattoo image:")
    up = files.upload()
    tattoo_img = Image.open(list(up.keys())[0]).convert("RGB")

elif image_source == "url":
    import requests
    from io import BytesIO

    # Replace these URLs with your own public-domain images
    BODY_URL = "https://images.unsplash.com/photo-1503104834685-7205e8607eb9?w=512"
    TATTOO_URL = "https://images.unsplash.com/photo-1611501275019-9b5cda994e8d?w=224"

    body_img = Image.open(BytesIO(requests.get(BODY_URL).content)).convert("RGB")
    tattoo_img = Image.open(BytesIO(requests.get(TATTOO_URL).content)).convert("RGB")

print(f"Body image:   {body_img.size} (W x H)")
print(f"Tattoo image: {tattoo_img.size} (W x H)")
show_images([body_img, tattoo_img], ["Body", "Tattoo"], figsize=(10, 5))

## 2. SAM2 Mask Extraction

Test `SkinMaskExtractor` with both `PointPrompt` and `BBoxPrompt`.

**VRAM**: ~3 GB for SAM2 hiera-large. The model is unloaded at the end of this section.

In [ ]:
#@title 2. SAM2 Mask Extraction {display-mode: "form"}
from sam2.sam2_image_predictor import SAM2ImagePredictor

# ── Load SAM2 ────────────────────────────────────────────────────────
print("Loading SAM2 (facebook/sam2-hiera-large)...")
sam_predictor = SAM2ImagePredictor.from_pretrained(
    "facebook/sam2-hiera-large",
)
print("SAM2 loaded.")
cleanup_vram()

extractor = SkinMaskExtractor(sam_predictor)

# ── PointPrompt ──────────────────────────────────────────────────────
w, h = body_img.size
point = PointPrompt(coords=[(w // 2, h // 2)])
print(f"\nPointPrompt: coords=({w // 2}, {h // 2})")

result_point = extractor.extract(body_img, point)
print(f"  mask shape : {result_point.mask.shape}")
print(f"  score      : {result_point.score:.4f}")
print(f"  True pixels: {result_point.mask.sum():,}")

# ── BBoxPrompt ────────────────────────────────────────────────────────
mx, my = w // 4, h // 4
bbox = BBoxPrompt(bbox=(mx, my, w - mx, h - my))
print(f"\nBBoxPrompt: bbox={bbox.bbox}")

result_bbox = extractor.extract(body_img, bbox)
print(f"  mask shape : {result_bbox.mask.shape}")
print(f"  score      : {result_bbox.score:.4f}")
print(f"  True pixels: {result_bbox.mask.sum():,}")

# ── Refine ────────────────────────────────────────────────────────────
print("\nRefining PointPrompt mask...")
result_refined = extractor.refine(body_img, point, result_point)
print(f"  refined score : {result_refined.score:.4f}")
print(f"  refined pixels: {result_refined.mask.sum():,}")

# ── Visualize ─────────────────────────────────────────────────────────
overlay_pt  = show_mask_overlay(body_img, result_point.mask)
overlay_bb  = show_mask_overlay(body_img, result_bbox.mask, color=(0, 255, 0))
overlay_ref = show_mask_overlay(body_img, result_refined.mask, color=(0, 0, 255))

show_images(
    [body_img, overlay_pt, overlay_bb, overlay_ref],
    [
        "Original",
        f"PointPrompt (score {result_point.score:.2f})",
        f"BBoxPrompt  (score {result_bbox.score:.2f})",
        f"Refined     (score {result_refined.score:.2f})",
    ],
    figsize=(20, 5),
)

# ── Keep mask for later sections ──────────────────────────────────────
sam_mask = result_point.mask  # bool ndarray (H, W)
print(f"\nSaved sam_mask: shape={sam_mask.shape}, dtype={sam_mask.dtype}")

# ── Unload SAM2 ──────────────────────────────────────────────────────
extractor.unload()
del sam_predictor, extractor
del result_point, result_bbox, result_refined
cleanup_vram()
print("SAM2 unloaded.")

## 3. Mask Post-processing

Test `dilate_mask` and `feather_mask` from `tatvton.utils.mask`.  
No GPU needed.

In [ ]:
#@title 3. Mask Post-processing {display-mode: "form"}

# ── Dilation ──────────────────────────────────────────────────────────
dilation_values = [0, 5, 10, 20]
dilated_masks = [dilate_mask(sam_mask, px) for px in dilation_values]

show_images(
    [mask_to_pil(m) for m in dilated_masks],
    [f"dilate={px}px" for px in dilation_values],
    figsize=(16, 4),
)

# ── Feathering (applied on top of 10px dilation) ─────────────────────
feather_values = [0.0, 3.0, 5.0, 10.0]
base_dilated = dilate_mask(sam_mask, 10)
feathered_masks = [feather_mask(base_dilated, sigma) for sigma in feather_values]

show_images(
    [Image.fromarray((m * 255).astype(np.uint8), mode="L") for m in feathered_masks],
    [f"feather sigma={s}" for s in feather_values],
    figsize=(16, 4),
)

# ── Verify properties ─────────────────────────────────────────────────
print("Dilation checks:")
for px, m in zip(dilation_values, dilated_masks):
    assert m.dtype == bool, f"Expected bool, got {m.dtype}"
    assert m.shape == sam_mask.shape
    print(f"  dilate={px:2d}px  dtype={m.dtype}  shape={m.shape}  "
          f"True pixels={m.sum():,}")

print("\nFeathering checks:")
for s, m in zip(feather_values, feathered_masks):
    assert m.dtype == np.float32, f"Expected float32, got {m.dtype}"
    assert 0.0 <= m.min() <= m.max() <= 1.0
    print(f"  sigma={s:4.1f}  dtype={m.dtype}  "
          f"range=[{m.min():.3f}, {m.max():.3f}]")

print("\nAll mask-processing checks passed.")

## 4. SDXL Inpainting + IP-Adapter

Test inpainting **without ControlNet** (the `khope/tatvton-controlnet-v1`
checkpoint is not yet on HF Hub).

Uses `StableDiffusionXLInpaintPipeline` from diffusers with IP-Adapter
for tattoo style guidance.

**VRAM**: ~8 GB with `enable_model_cpu_offload()`.

In [ ]:
#@title 4. SDXL Inpainting + IP-Adapter {display-mode: "form"}
from diffusers import StableDiffusionXLInpaintPipeline

# ── Config ────────────────────────────────────────────────────────────
SDXL_MODEL   = "stabilityai/stable-diffusion-xl-base-1.0"
IPA_REPO     = "h94/IP-Adapter"
IPA_SUBFOLDER = "sdxl_models"
IPA_WEIGHT   = "ip-adapter-plus_sdxl_vit-h.safetensors"
RESOLUTION   = 1024
SEED         = 42

# ── Load pipeline ─────────────────────────────────────────────────────
print("Loading SDXL pipeline (this may take a few minutes on first run)...")
pipe = StableDiffusionXLInpaintPipeline.from_pretrained(
    SDXL_MODEL,
    torch_dtype=torch.float16,
    variant="fp16",
)

print("Loading IP-Adapter...")
pipe.load_ip_adapter(
    IPA_REPO,
    subfolder=IPA_SUBFOLDER,
    weight_name=IPA_WEIGHT,
)
pipe.set_ip_adapter_scale(0.6)
pipe.enable_model_cpu_offload()
print("Pipeline ready.")
cleanup_vram()

# ── Prepare inputs ────────────────────────────────────────────────────
body_resized = resize_for_pipeline(body_img, RESOLUTION)

mask_pil_full = mask_to_pil(sam_mask)
mask_pil_resized = mask_pil_full.resize(
    (RESOLUTION, RESOLUTION), Image.Resampling.NEAREST,
)
mask_resized_bool = np.asarray(mask_pil_resized) > 127

tattoo_clip = tattoo_img.resize((224, 224), Image.Resampling.LANCZOS)

print(f"Body   (resized): {body_resized.size}")
print(f"Mask   (resized): {mask_pil_resized.size}, mode={mask_pil_resized.mode}")
print(f"Tattoo (for CLIP): {tattoo_clip.size}")

# ── Run inpainting ────────────────────────────────────────────────────
generator = torch.Generator(device="cpu").manual_seed(SEED)

print("Running inpainting (30 steps)...")
inpainted_result = pipe(
    prompt="a photorealistic tattoo on skin, high quality, detailed",
    image=body_resized,
    mask_image=mask_pil_resized,
    ip_adapter_image=tattoo_clip,
    strength=0.85,
    guidance_scale=7.5,
    num_inference_steps=30,
    generator=generator,
).images[0]

print(f"Output: {inpainted_result.size}, mode={inpainted_result.mode}")

# ── Visualize ─────────────────────────────────────────────────────────
show_images(
    [body_resized, mask_pil_resized, tattoo_clip, inpainted_result],
    ["Body (1024)", "Mask", "Tattoo (224)", "Inpainted"],
    figsize=(20, 5),
)

# ── Cleanup VRAM (keep inpainted_result for compositing) ─────────────
del pipe
cleanup_vram()
print("SDXL pipeline unloaded.")

## 5. Compositing

Test `Compositor` alpha blending with the inpainted result.  
No GPU needed.

In [ ]:
#@title 5. Compositing {display-mode: "form"}

# ── Compare different dilation / feather combos ────────────────────────
param_sets = [
    {"dilation_pixels": 0,  "feather_sigma": 0.0},   # no processing
    {"dilation_pixels": 5,  "feather_sigma": 3.0},   # light
    {"dilation_pixels": 10, "feather_sigma": 5.0},   # default
    {"dilation_pixels": 20, "feather_sigma": 10.0},  # heavy
]

composited_results = []
for cfg in param_sets:
    comp = Compositor(**cfg)
    result = comp.composite(body_resized, inpainted_result, mask_resized_bool)
    composited_results.append(result)
    print(f"  dilation={cfg['dilation_pixels']:2d}, "
          f"feather={cfg['feather_sigma']:4.1f} -> "
          f"{result.size}, mode={result.mode}")

show_images(
    composited_results,
    [f"d={c['dilation_pixels']}, s={c['feather_sigma']}" for c in param_sets],
    figsize=(20, 5),
)

# ── composite_with_resize: back to original resolution ────────────────
comp_default = Compositor(dilation_pixels=10, feather_sigma=5.0)
final_result = comp_default.composite_with_resize(
    original_full=body_img,
    inpainted=inpainted_result,
    mask=mask_resized_bool,
    original_resized=body_resized,
)

print(f"\nFinal result (original resolution): {final_result.size}")
show_images(
    [body_img, final_result],
    ["Original", "Final (original res)"],
    figsize=(12, 6),
)

## 6. End-to-End Integration

Combine all steps into a single flow:
**SAM2 -> Mask Processing -> SDXL Inpainting -> Compositing**

SAM2 and SDXL are loaded sequentially (never simultaneously) to stay within
the T4 16 GB VRAM budget.

**VRAM peak**: ~8 GB.

In [ ]:
#@title 6a. End-to-End Integration {display-mode: "form"}
from sam2.sam2_image_predictor import SAM2ImagePredictor
from diffusers import StableDiffusionXLInpaintPipeline

# ── Use TatVTONConfig defaults as reference ───────────────────────────
config = TatVTONConfig()
print("TatVTONConfig defaults:")
print(f"  resolution       : {config.resolution}")
print(f"  steps            : {config.num_inference_steps}")
print(f"  guidance_scale   : {config.guidance_scale}")
print(f"  strength         : {config.strength}")
print(f"  ip_adapter_scale : {config.ip_adapter_scale}")
print(f"  mask_dilation    : {config.mask_dilation_pixels}px")
print(f"  mask_feather     : {config.mask_feather_sigma}")

# ── Phase 1: SAM2 Mask Extraction ─────────────────────────────────────
print("\n--- Phase 1: SAM2 Mask Extraction ---")
predictor = SAM2ImagePredictor.from_pretrained(config.sam_model_id)
extractor = SkinMaskExtractor(predictor)

w, h = body_img.size
region = PointPrompt(coords=[(w // 2, h // 2)])
mask_result = extractor.extract(body_img, region)
print(f"  score: {mask_result.score:.4f}, pixels: {mask_result.mask.sum():,}")

extractor.unload()
del predictor, extractor
cleanup_vram()

# ── Phase 2: Resize ───────────────────────────────────────────────────
print("\n--- Phase 2: Resize ---")
body_r = resize_for_pipeline(body_img, config.resolution)
mask_r = np.array(
    mask_to_pil(mask_result.mask).resize(
        (config.resolution, config.resolution), Image.Resampling.NEAREST,
    )
) > 127
mask_r_pil = mask_to_pil(mask_r)
tattoo_r = tattoo_img.resize((224, 224), Image.Resampling.LANCZOS)
print(f"  body={body_r.size}, mask={mask_r_pil.size}, tattoo={tattoo_r.size}")

# ── Phase 3: Inpainting (without ControlNet) ──────────────────────────
print("\n--- Phase 3: SDXL Inpainting ---")
pipe = StableDiffusionXLInpaintPipeline.from_pretrained(
    config.sdxl_model_id,
    torch_dtype=torch.float16,
    variant="fp16",
)
pipe.load_ip_adapter(
    config.ip_adapter_repo_id,
    subfolder="sdxl_models",
    weight_name=config.ip_adapter_weight_name,
)
pipe.set_ip_adapter_scale(config.ip_adapter_scale)
pipe.enable_model_cpu_offload()

generator = torch.Generator(device="cpu").manual_seed(42)
inpainted = pipe(
    prompt="a photorealistic tattoo on skin, high quality, detailed",
    image=body_r,
    mask_image=mask_r_pil,
    ip_adapter_image=tattoo_r,
    strength=config.strength,
    guidance_scale=config.guidance_scale,
    num_inference_steps=config.num_inference_steps,
    generator=generator,
).images[0]
print(f"  inpainted: {inpainted.size}")

del pipe
cleanup_vram()

# ── Phase 4: Compositing ──────────────────────────────────────────────
print("\n--- Phase 4: Compositing ---")
compositor = Compositor(
    dilation_pixels=config.mask_dilation_pixels,
    feather_sigma=config.mask_feather_sigma,
)
final = compositor.composite_with_resize(
    original_full=body_img,
    inpainted=inpainted,
    mask=mask_r,
    original_resized=body_r,
)
print(f"  final: {final.size}")

# ── Final visualization ───────────────────────────────────────────────
show_images(
    [body_img, mask_to_pil(mask_result.mask), tattoo_img, final],
    ["Input Body", "SAM2 Mask", "Tattoo Design", "Final Result"],
    figsize=(20, 5),
)

print("\n=== All components verified successfully ===")

In [ ]:
#@title 6b. Parameter Experiments {display-mode: "form"}

# Compare compositing parameters using the inpainted output from 6a.
experiment_params = [
    {"dilation_pixels": 5,  "feather_sigma": 2.0},
    {"dilation_pixels": 10, "feather_sigma": 5.0},
    {"dilation_pixels": 15, "feather_sigma": 8.0},
    {"dilation_pixels": 20, "feather_sigma": 12.0},
]

exp_results = []
for params in experiment_params:
    c = Compositor(**params)
    r = c.composite_with_resize(body_img, inpainted, mask_r, body_r)
    exp_results.append(r)

show_images(
    exp_results,
    [
        f"d={p['dilation_pixels']}, sigma={p['feather_sigma']}"
        for p in experiment_params
    ],
    figsize=(20, 5),
)

print("Parameter sweep complete.")